# Notebook 1 — Data Collection (Crawling → Screening → Scraping)

Notebook ini **berdiri sendiri** (tanpa import `.py`) dan ditulis **sederhana** (tanpa class).
Dibagi **3 bagian** sesuai flow data collection:

| Bagian | Nama | Yang dilakukan | Output |
|--------|------|----------------|--------|
| 1 | **CRAWLING** | telusuri indeks per tanggal, kumpulkan link **semua** artikel | `news_links` (semua) |
| 2 | **SCREENING** | saring judul pakai kata kunci → tandai kandidat (+ 2% kontrol negatif) | kolom `is_candidate` |
| 3 | **SCRAPING** | unduh HTML kandidat (`.gz`) + ekstrak judul/tanggal/isi | `html .gz` + `news_clean_raw` |

Class HTML sudah ditemukan di Notebook 0: judul `detail__title`, tanggal `meta publishdate`,
isi `detail__body-text`.

Install: `pip install requests beautifulsoup4 lxml pandas pyarrow openpyxl tqdm`

## 0. Import & pengaturan

In [ ]:
import re, time, gzip, random
from datetime import date, timedelta
from pathlib import Path
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

BASE_URL = "https://finance.detik.com/indeks"
HEADERS  = {"User-Agent": "Mozilla/5.0 (research crawler; contact: GANTI@email.anda)"}
DELAY    = 1.0           # jeda antar-request (detik) — etika crawling

RAW     = Path("data/raw")
INTERIM = Path("data/interim")
RAW.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)
random.seed(42)   # agar sampel kontrol-negatif selalu sama (reprodusibel)

def ambil_halaman(url):
    """Unduh satu URL, lalu jeda sejenak (etika). Kembalikan HTML string."""
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    time.sleep(DELAY + random.uniform(0.3, 1.0))
    return r.text

def simpan_html(html, path):
    """Simpan HTML sebagai file gzip (hemat ruang)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(gzip.compress(html.encode("utf-8")))

def id_dari_url(url):
    """Angka setelah 'd-' sebagai id artikel. .../d-8688622/... -> '8688622'."""
    return re.findall(r"/d-(\d+)/", url)[0]

def subkanal_dari_url(url):
    """.../moneter/d-123/... -> 'moneter'."""
    hasil = re.findall(r"finance\.detik\.com/([a-z0-9-]+)/d-\d+", url)
    return hasil[0] if hasil else None

# BAGIAN 1 — CRAWLING

Telusuri halaman indeks DetikFinance per tanggal, kumpulkan **daftar link + judul**
semua artikel. Ini **murah** (cuma ambil daftar ringkas dari halaman indeks, belum HTML
lengkap tiap artikel).

Output: `news_links.parquet` berisi semua artikel (belum disaring).

In [ ]:
def parse_indeks(html):
    """Dari HTML halaman indeks -> daftar artikel (loop sederhana)."""
    soup = BeautifulSoup(html, "html.parser")
    data = []
    for artikel in soup.find_all("article", class_="list-content__item"):
        link_tag = artikel.find("a", class_="media__link")
        if not link_tag:
            continue
        url = link_tag["href"].split("?")[0]
        if "/d-" not in url:
            continue
        judul_tag = artikel.find("h3", class_="media__title") or artikel.find(class_="media__title")
        tgl_tag   = artikel.find(class_="media__date")
        data.append({
            "article_id": id_dari_url(url),
            "url": url,
            "title": judul_tag.text.strip() if judul_tag else "",
            "subchannel": subkanal_dari_url(url),
            "date_text": tgl_tag.text.strip() if tgl_tag else None,
        })
    return data

def crawling(tgl_awal, tgl_akhir):
    """Kumpulkan link semua artikel dari indeks, tanggal demi tanggal."""
    # buat daftar tanggal
    tanggal_list = []
    t = tgl_awal
    while t <= tgl_akhir:
        tanggal_list.append(t)
        t += timedelta(days=1)

    semua_link = []
    for tgl in tqdm(tanggal_list, desc="crawling indeks"):
        for halaman in range(1, 50):
            url_indeks = f"{BASE_URL}?page={halaman}&date={tgl:%m/%d/%Y}"
            html = ambil_halaman(url_indeks)
            if halaman == 1:
                simpan_html(html, RAW / "html_index" / f"{tgl:%Y/%m/%d}" / f"page_{halaman}.html.gz")
            daftar = parse_indeks(html)
            if not daftar:        # halaman kosong -> tanggal ini selesai
                break
            for art in daftar:
                art["index_date"] = tgl.isoformat()
            semua_link.extend(daftar)
    return pd.DataFrame(semua_link)

In [ ]:
# Jalankan CRAWLING (uji 1 minggu dulu; untuk penuh ganti ke date(2026, 9, 30))
links = crawling(date(2024, 1, 2), date(2024, 1, 8))
print("total link terkumpul:", len(links))
links.head()

# BAGIAN 2 — SCREENING

Saring artikel berdasarkan **judul**: apakah memuat kata kunci bank/kebijakan/pasar?
Yang lolos = **kandidat** (HTML-nya akan diunduh di Bagian 3).

**Kontrol negatif (neg_control):** kita juga menandai **2% acak** dari artikel yang
TIDAK lolos, untuk diperiksa manual nanti. Gunanya: mengukur apakah filter kita
melewatkan berita yang sebenarnya relevan (ukuran *recall* filter).

In [ ]:
# kata kunci (huruf kecil) untuk dicek di judul
KEYWORDS = [
    "bbca", "bca", "bank central asia",
    "bbri", "bri", "bank rakyat",
    "bbni", "bni", "bank negara",
    "bmri", "bank mandiri", "mandiri",
    "himbara", "bank bumn", "perbankan",
    "ojk", "bi rate", "suku bunga", "lps", "kur", "npl", "kredit", "dividen",
    "ihsg", "rupiah", "inflasi",
]
BURSA_SUBCHANNELS = ["bursa-dan-valas", "market-research"]
NEG_SAMPLE_RATE   = 0.02   # 2% non-kandidat jadi kontrol negatif

def lolos_keyword(judul, subkanal):
    """True kalau judul memuat kata kunci / subkanal bursa."""
    if subkanal in BURSA_SUBCHANNELS:
        return True
    judul_kecil = (judul or "").lower()
    return any(kata in judul_kecil for kata in KEYWORDS)

def screening(df_links):
    """Tambah kolom is_candidate & neg_control."""
    is_cand, neg = [], []
    for _, row in df_links.iterrows():
        lolos = lolos_keyword(row["title"], row["subchannel"])
        is_neg = False
        if not lolos and random.random() < NEG_SAMPLE_RATE:
            lolos  = True      # ikut diunduh (untuk dicek manual)
            is_neg = True      # tapi ditandai sbg kontrol negatif
        is_cand.append(lolos)
        neg.append(is_neg)
    df = df_links.copy()
    df["is_candidate"] = is_cand
    df["neg_control"]  = neg
    return df

In [ ]:
links = screening(links)
print("total link       :", len(links))
print("kandidat (lolos)  :", int(links.is_candidate.sum()))
print("  - kata kunci    :", int((links.is_candidate & ~links.neg_control).sum()))
print("  - kontrol neg.  :", int(links.neg_control.sum()))
print("tidak lolos       :", int((~links.is_candidate).sum()))

In [ ]:
# LIHAT 2% yang TIDAK lolos tapi diambil sbg kontrol negatif
# (judul-judul ini TIDAK memuat kata kunci — cek: apakah ada yg sebenarnya relevan?)
print("=== Sampel kontrol negatif (judul yang tidak lolos screening) ===")
links[links.neg_control][["title", "subchannel", "url"]].head(10)

In [ ]:
# Bandingkan: contoh judul yang LOLOS (kandidat sejati) — harus jelas soal bank/pasar
print("=== Sampel kandidat (judul yang lolos screening) ===")
links[links.is_candidate & ~links.neg_control][["title", "subchannel"]].head(10)

# BAGIAN 3 — SCRAPING

Untuk setiap **kandidat**: unduh HTML lengkap, simpan `.gz`, lalu ekstrak judul/tanggal/isi.
Inilah yang "berat" (download isi penuh), makanya hanya dilakukan untuk kandidat.

In [ ]:
def parse_artikel(html, url):
    """Dari HTML satu artikel -> judul, tanggal, isi (class dari Notebook 0)."""
    soup = BeautifulSoup(html, "html.parser")
    # 1. judul
    h1 = soup.find("h1", class_="detail__title")
    judul = h1.text.strip() if h1 else ""
    # 2. tanggal
    meta = soup.find("meta", attrs={"name": "publishdate"})
    tanggal = meta["content"] if meta else None
    # 3. isi: gabung semua <p> dalam body
    body = soup.find("div", class_="detail__body-text")
    isi = ""
    if body:
        paragraf = [p.text.strip() for p in body.find_all("p") if p.text.strip()]
        isi = " ".join(paragraf)
    return {
        "article_id": id_dari_url(url), "url": url, "title": judul,
        "published_at": tanggal, "subchannel": subkanal_dari_url(url),
        "n_words": len(isi.split()), "body_text": isi,
    }

def scraping(df_links):
    """Unduh HTML + ekstrak isi untuk baris yang is_candidate=True."""
    kandidat = df_links[df_links.is_candidate]
    html_saved = {}
    artikel_bersih = []
    for _, row in tqdm(kandidat.iterrows(), total=len(kandidat), desc="scraping kandidat"):
        try:
            html = ambil_halaman(row["url"] + "?single=1")
            aid  = row["article_id"]
            simpan_html(html, RAW / "html_article" / aid[:3] / f"{aid}.html.gz")
            html_saved[aid] = True
            isi = parse_artikel(html, row["url"])
            isi["index_date"] = row["index_date"]
            artikel_bersih.append(isi)
        except Exception as e:
            print("  gagal:", row["url"], "->", e)
    # tandai html_saved di tabel link
    df = df_links.copy()
    df["html_saved"] = df["article_id"].map(html_saved).fillna(False)
    return df, pd.DataFrame(artikel_bersih)

In [ ]:
links, bersih = scraping(links)

# simpan ketiga output
links.to_parquet(INTERIM / "news_links.parquet", index=False)
links.to_excel(INTERIM / "news_links.xlsx", index=False)
bersih.to_parquet(INTERIM / "news_clean_raw.parquet", index=False)

print("html tersimpan   :", int(links.html_saved.sum()))
print("artikel ter-ekstrak:", len(bersih))
bersih.head()

## Verifikasi akhir

In [ ]:
print("total link        :", len(links))
print("kandidat          :", int(links.is_candidate.sum()))
print("  kontrol negatif :", int(links.neg_control.sum()))
print("html tersimpan    :", int(links.html_saved.sum()))
print("artikel ter-ekstrak:", len(bersih))
print("\ncontoh isi berita:")
print("judul   :", bersih.iloc[0].title)
print("tanggal :", bersih.iloc[0].published_at)
print("isi     :", bersih.iloc[0].body_text[:200])

## Catatan
- **RENTANG PENUH:** ganti `date(2024, 1, 8)` di Bagian 1 jadi `date(2026, 9, 30)` (bisa 6-11 jam).
- Versi ini tanpa checkpoint/retry (untuk belajar). Versi tahan-putus ada di `old/`.
- **neg_control** = 2% non-kandidat yang diunduh untuk cek manual: apakah filter melewatkan
  berita relevan? Periksa `links[links.neg_control]` secara manual untuk mengukur recall filter.